# Modelado — forecast de **gas** (`prod_gas`, t+1)

Análogo al `03_modeling.ipynb` (que modela petróleo), pero con **target = `prod_gas`**
del mes siguiente (`y_next`). Es el **segundo modelo** que pidió la cátedra (ADR-042):
un modelo por producción, mismo encuadre.

Comparación de **regresión lineal (Ridge)**, **Random Forest** y **XGBoost**.

**Protocolo (ADR-028, reutilizado):**
- Se **entrena en `train`** y se **compara en `val`**; `test` queda intacto.
- Métricas: **RMSE** (m³, penaliza errores grandes) y **R²** (varianza explicada).
- Todo el preprocesamiento (one-hot, imputación, normalización) vive en el `Pipeline`
  y se **ajusta solo con el train** de cada fold → sin leakage val→train.

La lógica reutilizable vive en `ml/` (parametrizada por `target`, ADR-042); acá solo
se orquesta pasando `target="prod_gas"`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from ml import modeling

TARGET = "prod_gas"  # lo único que cambia respecto del notebook 03 (petróleo)
pd.set_option("display.width", 120)

## 1. Dataset y matriz de features

`build_feature_matrix(target="prod_gas")` arma el dataset del forecast de gas
(universo **gasífero** definido solo con train, target `prod_gas` del mes t+1) y
devuelve las features **crudas** (numéricas + categóricas sin codificar). El one-hot
NO se hace acá: vive en el `Pipeline` para poder ajustarlo por fold durante la CV.

Las features = todas las columnas menos las claves (`idpozo`, `periodo`,
`periodo_objetivo`, `split`) y el target `y_next`.

In [ ]:
ds_enc, feature_cols = modeling.build_feature_matrix(target=TARGET)

print(f"Filas: {len(ds_enc):,}  |  Features: {len(feature_cols)}")
print(ds_enc["split"].value_counts().reindex(["train", "val", "test"]).to_string())

In [ ]:
# Las features de ingeniería autorregresivas deben estar calculadas sobre GAS:
eng = [c for c in feature_cols if c.startswith("prod_gas_")
       or c in ("water_cut", "produjo_mes_pasado", "prod_vecinos_mean")]
print("Features de ingeniería (sobre prod_gas):")
print(eng)

### Nota anti-leakage (importante para el informe)

Conviene distinguir **dos tipos** de leakage y por qué este pipeline está cubierto:

1. **Estadísticos que aprenden un parámetro** del dataset — imputación (mediana),
   one-hot (vocabulario), escalado (media/desvío). Estos **sí** deben ajustarse
   **solo en train**: por eso viven dentro del `Pipeline` (`ml/preprocessing.py`) y
   `cross_val_score` los **reajusta con el train de cada fold**.

2. **Features de ingeniería** (`prod_gas_roll3`, `prod_gas_acum6`, `prod_gas_lag12`,
   `prod_gas_delta1`, `prod_vecinos_mean`) — **no** ajustan ningún parámetro global.
   Cada fila se calcula **solo con datos de su propio mes `t` o anteriores**:
   - los lags/medias móviles/acumulados usan la historia del **mismo pozo** hasta `t`
     (lag **por calendario**, no `shift` de filas);
   - el **promedio de vecinos** en `(pozo, mes t)` usa la producción de los vecinos
     **en ese mismo mes t** (≤ t), y el conjunto de vecinos se define con
     **coordenadas estáticas** (no con el target).

   Por eso calcularlas sobre todo el histórico **no es leakage**: no son un
   estadístico aprendido del conjunto, son una agregación por fila que solo mira el
   pasado. El leakage aparecería si una feature usara una media **global** (todos los
   pozos y meses) ajustada sobre el dataset entero — ninguna lo hace.

## 2. Split train/val + preprocesamiento (anti-leakage)

- **Imputación** de NaN (mediana de train para físicas/operativa; 0 + flag para
  volúmenes) — los árboles y la regresión no toman NaN. Los NaN vienen de las
  features de historia (primeros meses de cada pozo).
- **Normalización** (media 0, desvío 1) **solo para la regresión lineal**; los
  árboles no la necesitan.

Ambos estadísticos se ajustan **solo en train** (dentro del `Pipeline`).

In [ ]:
X_train, y_train, X_val, y_val = modeling.split_train_val(ds_enc, feature_cols)
print("train:", X_train.shape, " | val:", X_val.shape)

## 3. Entrenamiento y comparación en val (sin tunear)

`train_eval_models` entrena los 3 modelos en train (cada uno en su `Pipeline`) y los
evalúa en val. Agregamos la **persistencia** (ŷ(t+1) = `prod_gas`(t)) como referencia:
es el baseline a batir (ADR-029).

> ⚠️ Entrena 3 modelos sobre ~270k filas: puede tardar un poco.

In [ ]:
tabla, modelos = modeling.train_eval_models(X_train, y_train, X_val, y_val)

pers = modeling.persistence_val(ds_enc, target=TARGET)
comparacion = pd.concat(
    [tabla, pd.DataFrame([{"modelo": "persistencia (baseline)", **pers}])],
    ignore_index=True,
).sort_values("val_rmse").reset_index(drop=True)
comparacion.round(3)

**Lectura.** _(completar al correr)_

- ¿Qué modelo gana sin tunear? ¿Supera a la persistencia de gas?
- Comparar con lo que pasó en petróleo (notebook 03): ahí XGBoost ganaba sin tunear
  y la persistencia era dura de batir por el R² alto (autoregresión fuerte).

## 4. Tuning de hiperparámetros con CV temporal

Igual que en petróleo: **no** se usa `KFold` (barajaría fechas → leakage
futuro→pasado), sino **CV temporal (ventana expansiva) por mes**
(`modeling.time_series_folds`). El `Pipeline` (imputación + one-hot [+ escalado] +
modelo) se reajusta **solo con el train de cada fold**. La búsqueda se hace **dentro
de train**; **val queda intacto** como comparación final.

Modelos: **Ridge** (tuneando `alpha`), **Random Forest** y **XGBoost**. Cada uno
muestra una barra de progreso (`tqdm`).

> ⚠️ Esta celda entrena muchos modelos: puede tardar varios minutos.

In [ ]:
tabla_cv, searches = modeling.tune_all(ds_enc, feature_cols, n_splits=4)
tabla_cv

In [ ]:
for nombre, search in searches.items():
    print(f"{nombre}: cv_rmse={-search.best_score_:.3f}")
    print("  ", search.best_params_)

### 4.1 Evaluación de los modelos tuneados en val

Cada `best_estimator_` (reentrenado en **todo** train con sus mejores hiperparámetros)
se evalúa sobre **val**, que no se tocó durante el tuning. Comparamos contra la
persistencia de gas.

In [ ]:
comparacion_tuned = modeling.eval_searches_on_val(searches, ds_enc, feature_cols, target=TARGET)
comparacion_tuned.round(3)

### 4.2 Métricas en train y val con la mejor config

Mide RMSE/R² en **train** y en **val** para la mejor config de cada modelo. El gap
train→val muestra el **sobreajuste**: si train es mucho mejor que val, el modelo
memoriza.

In [ ]:
metricas_train_val = modeling.eval_searches_train_val(searches, ds_enc, feature_cols, target=TARGET)
metricas_train_val.round(3)

## 5. Conclusión _(completar al correr)_

- **Campeón de gas** (menor `val_rmse` tuneado) y si **supera la persistencia**
  (vara de éxito, ADR-029).
- Mejores hiperparámetros del campeón → registrarlos (análogo a `BEST_PARAMS` /
  ADR-040 para petróleo) para el `train.py --target prod_gas`.
- El **juicio final** se hace **una sola vez** sobre `test`
  (`python -m ml.train --target prod_gas --final`).